In [ ]:
# Cell 1: Colab T4 dependencies, Drive persistence, and configuration
# Colab's active kernel is used directly; no separate virtualenv is required.
ENV_NAME = 'v'
!pip -q install -U 'ultralytics>=8.3.0,<9' 'transformers>=4.45,<5' accelerate opencv-contrib-python-headless pandas scipy matplotlib tqdm lxml
import os, sys, json, math, csv, shutil, subprocess, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Persist outputs in Drive when running in Colab. Set USE_GOOGLE_DRIVE=0 for local/temporary storage.
USE_GOOGLE_DRIVE = os.environ.get('USE_GOOGLE_DRIVE', '1') == '1'
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        STORAGE_ROOT = Path('/content/drive/MyDrive/adas_monocular_results')
    except ImportError:
        STORAGE_ROOT = Path('/content/adas_results')
else:
    STORAGE_ROOT = Path('/content/adas_results')
ROOT = STORAGE_ROOT
ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH = ROOT.parent / 'complete_results.zip'
INPUT_VIDEO = os.environ.get('INPUT_VIDEO', '/content/input.mp4')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HALF = DEVICE == 'cuda'
MAX_FRAMES = int(os.environ.get('MAX_FRAMES', '0'))
FRAME_STRIDE = int(os.environ.get('FRAME_STRIDE', '1'))
MODEL_SELECTION = [
 {'stage':'depth','selected':'Depth Anything V2 Small (Hugging Face)','repository':'https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf','checkpoint':'depth-anything/Depth-Anything-V2-Small-hf','input':'RGB frame','output':'relative inverse-depth map','python':'3.9+','pytorch':'2.x','cuda':'optional','compile':'none','t4':'yes','reason':'maintained Transformers checkpoint with no custom CUDA'},
 {'stage':'vehicles','selected':'Ultralytics YOLO11m','repository':'https://github.com/ultralytics/ultralytics','checkpoint':'yolo11m.pt','input':'RGB frame','output':'2D COCO detections','python':'3.8+','pytorch':'2.x','cuda':'optional','compile':'none','t4':'yes','reason':'official downloadable checkpoint and stable API'},
 {'stage':'tracking','selected':'IoU/appearance-independent track manager','repository':'https://github.com/ultralytics/ultralytics','checkpoint':'none','input':'detections','output':'persistent IDs','python':'3.8+','pytorch':'none','cuda':'none','compile':'none','t4':'yes','reason':'dependency-free deterministic tracker avoids native build issues'},
 {'stage':'lane','selected':'Road segmentation plus evidence-based lane marking geometry','repository':'https://huggingface.co/nvidia/segformer-b2-finetuned-cityscapes-1024-1024','checkpoint':'nvidia/segformer-b2-finetuned-cityscapes-1024-1024','input':'RGB frame','output':'road mask; lane pixels from image evidence','python':'3.9+','pytorch':'2.x','cuda':'optional','compile':'none','t4':'yes','reason':'road mask is pretrained; lane markings are fitted only from observed pixels'},
 {'stage':'vo','selected':'OpenCV essential-matrix monocular VO','repository':'https://opencv.org','checkpoint':'none','input':'successive RGB frames','output':'relative camera rotations/translations','python':'3.8+','pytorch':'none','cuda':'none','compile':'none','t4':'yes','reason':'portable geometric baseline with explicit scale ambiguity'},
 {'stage':'opendrive','selected':'OpenDRIVE XML writer and structural validator','repository':'https://www.asam.net/standards/detail/opendrive/','checkpoint':'none','input':'reconstructed geometry','output':'road.xodr','python':'3.8+','pytorch':'none','cuda':'none','compile':'none','t4':'yes','reason':'deterministic standards output from measured/estimated geometry'}
]
print(json.dumps({'device':DEVICE,'torch':torch.__version__,'cuda':torch.version.cuda,'input_video':INPUT_VIDEO,'results_dir':str(ROOT),'archive':str(ARCHIVE_PATH),'models':MODEL_SELECTION}, indent=2))

In [ ]:
# Cell 2: input metadata and actual model initialization
from transformers import AutoImageProcessor, AutoModelForDepthEstimation, SegformerImageProcessor, SegformerForSemanticSegmentation
from ultralytics import YOLO
if not Path(INPUT_VIDEO).exists(): raise FileNotFoundError(f'Upload an MP4 to {INPUT_VIDEO} or set INPUT_VIDEO')
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened(): raise RuntimeError('Input video could not be opened')
FPS = float(cap.get(cv2.CAP_PROP_FPS) or 0); WIDTH=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); HEIGHT=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if FPS <= 0 or WIDTH <= 0 or HEIGHT <= 0: raise RuntimeError('Input video metadata is invalid')
DEPTH_PROCESSOR = AutoImageProcessor.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf')
DEPTH_MODEL = AutoModelForDepthEstimation.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf').to(DEVICE).eval()
ROAD_PROCESSOR = SegformerImageProcessor.from_pretrained('nvidia/segformer-b2-finetuned-cityscapes-1024-1024')
ROAD_MODEL = SegformerForSemanticSegmentation.from_pretrained('nvidia/segformer-b2-finetuned-cityscapes-1024-1024').to(DEVICE).eval()
VEHICLE_MODEL = YOLO('yolo11m.pt')
COCO_VEHICLES = {'car','truck','bus','motorcycle','bicycle','person'}
print({'fps':FPS,'resolution':(WIDTH,HEIGHT),'frames':TOTAL,'duration_s':TOTAL/FPS,'device':DEVICE})

In [ ]:
# Cell 3: depth, road mask, and image-evidence camera calibration helpers
from PIL import Image
def infer_depth(frame):
    rgb=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB); inputs=DEPTH_PROCESSOR(images=Image.fromarray(rgb), return_tensors='pt').to(DEVICE)
    with torch.inference_mode(): pred=DEPTH_MODEL(**inputs).predicted_depth
    depth=torch.nn.functional.interpolate(pred.unsqueeze(1), size=frame.shape[:2], mode='bicubic', align_corners=False).squeeze().float().cpu().numpy()
    depth=(depth-np.nanpercentile(depth,1))/(np.nanpercentile(depth,99)-np.nanpercentile(depth,1)+1e-6)
    return np.clip(depth,0,1), {'status':'MEASURED','metric':False,'source':'Depth Anything V2 relative inverse-depth','unit':'relative'}
def infer_road_mask(frame):
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB); inp=ROAD_PROCESSOR(images=Image.fromarray(rgb),return_tensors='pt').to(DEVICE)
    with torch.inference_mode(): logits=ROAD_MODEL(**inp).logits
    logits=torch.nn.functional.interpolate(logits,size=frame.shape[:2],mode='bilinear',align_corners=False)[0]
    # Cityscapes class 0 is road in this checkpoint's id2label mapping.
    mask=(logits.argmax(0).cpu().numpy()==0).astype(np.uint8)
    return mask
def estimate_calibration(frame, road_mask):
    h,w=frame.shape[:2]; ys,xs=np.where(road_mask & (np.indices((h,w))[0]>int(.45*h)))
    horizon=float(np.percentile(ys,5)) if len(ys)>100 else float('nan')
    # Vanishing-point/focal estimates are only emitted when two strong line families exist.
    edges=cv2.Canny(frame,80,180); lines=cv2.HoughLinesP(edges,1,np.pi/180,threshold=80,minLineLength=max(30,w//12),maxLineGap=20)
    slopes=[]
    if lines is not None:
      for x1,y1,x2,y2 in lines[:,0]:
        if abs(x2-x1)>5: slopes.append((y2-y1)/(x2-x1))
    enough=len(slopes)>=4 and np.isfinite(horizon)
    params={'fx':{'value':None,'unit':'px','status':'UNKNOWN','confidence':0,'uncertainty':None,'source':'not identifiable from single unconstrained view'},'fy':{'value':None,'unit':'px','status':'UNKNOWN','confidence':0,'uncertainty':None,'source':'not identifiable from single unconstrained view'},'cx':{'value':w/2,'unit':'px','status':'ASSUMED','confidence':0.1,'uncertainty':w/2,'source':'image center prior'},'cy':{'value':h/2,'unit':'px','status':'ASSUMED','confidence':0.1,'uncertainty':h/2,'source':'image center prior'},'horizon_y':{'value':horizon if np.isfinite(horizon) else None,'unit':'px','status':'ESTIMATED' if enough else 'UNKNOWN','confidence':0.35 if enough else 0,'uncertainty':max(5,h*.05) if enough else None,'source':'road-mask lower-image evidence'},'roll':{'value':None,'unit':'rad','status':'UNKNOWN','confidence':0,'uncertainty':None,'source':'insufficient constraints'}}
    return params
print('Calibration is evidence-gated; no focal length or metric camera height is hardcoded.')

In [ ]:
# Cell 4: sequence lane perception from pretrained road mask + observed lane-marking pixels
def lane_points(frame, road_mask, depth):
    h,w=frame.shape[:2]; hsv=cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
    bright=(hsv[:,:,2]>150)&(hsv[:,:,1]<90); yellow=(hsv[:,:,0]>12)&(hsv[:,:,0]<42)&(hsv[:,:,1]>60)&(hsv[:,:,2]>100)
    evidence=(bright|yellow)&(road_mask>0); rows=np.where(evidence & (np.indices((h,w))[0]>int(.52*h)))[0]
    candidates=[]
    for side in ('left','right'):
      xs=[]; ys=[]; zs=[]
      for y in np.linspace(.55*h,.97*h,18).astype(int):
        xx=np.where(evidence[y])[0]; xx=xx[xx<w//2] if side=='left' else xx[xx>=w//2]
        if len(xx):
          x=int(np.median(xx)); xs.append(x); ys.append(y); zs.append(float(depth[y,x]))
      if len(xs)>=5:
        coef=np.polyfit(np.array(ys)/h,np.array(xs)/w,2); candidates.append({'side':side,'points_px':list(zip(xs,ys)),'fit_coeff_px_norm':coef.tolist(),'depth_relative_median':float(np.median(zs)),'status':'MEASURED','source':'road segmentation + lane marking pixels'})
    return candidates
def lane_rows(frame_id,timestamp,lanes):
    out=[]
    for i,lane in enumerate(lanes):
      for x,y in lane['points_px']:
        out.append({'frame':frame_id,'timestamp_s':timestamp,'lane_id':i+1,'side':lane['side'],'x_px':x,'y_px':y,'z_relative':lane['depth_relative_median'],'status':'MEASURED','metric_status':'UNKNOWN','confidence_source':'observed pixels'})
    return out
print('Lane counts are per-frame observed boundary counts; missing boundaries remain missing.')

In [ ]:
# Cell 5: monocular VO, temporal fusion, and honest speed/scale handling
def vo_step(prev,cur,K):
    a=cv2.cvtColor(prev,cv2.COLOR_BGR2GRAY); b=cv2.cvtColor(cur,cv2.COLOR_BGR2GRAY); p=cv2.goodFeaturesToTrack(a,1200,0.01,8)
    if p is None or len(p)<20: return None
    q,st,_=cv2.calcOpticalFlowPyrLK(a,b,p,None); p=p[st[:,0]==1]; q=q[st[:,0]==1]
    if len(p)<12: return None
    E,mask=cv2.findEssentialMat(p,q,K,cv2.RANSAC,.999,1.0);
    if E is None: return None
    _,R,t,mask=cv2.recoverPose(E,p,q,K)
    return R,t.reshape(3),int(mask.sum())
def process_video():
    cap=cv2.VideoCapture(INPUT_VIDEO); rows=[]; lane_rows_all=[]; depth_stats=[]; frames=[]; prev=None; pose=np.eye(4); poses=[]
    first=None; calibration=None; frame_i=0; processed=0; failed_vo=0
    while True:
      ok,frame=cap.read();
      if not ok or (MAX_FRAMES and processed>=MAX_FRAMES): break
      if frame_i%FRAME_STRIDE: frame_i+=1; continue
      if first is None: first=frame.copy()
      depth,dmeta=infer_depth(frame); road=infer_road_mask(frame);
      if calibration is None: calibration=estimate_calibration(frame,road)
      lanes=lane_points(frame,road,depth); lane_rows_all += lane_rows(frame_i,frame_i/FPS,lanes)
      if prev is not None:
        K=np.array([[WIDTH,0,WIDTH/2],[0,WIDTH,HEIGHT/2],[0,0,1]],dtype=float); step=vo_step(prev,frame,K)
        if step is not None:
          R,t,inliers=step; T=np.eye(4); T[:3,:3]=R; T[:3,3]=t; pose=pose@np.linalg.inv(T)
        else: failed_vo+=1
      poses.append({'frame':frame_i,'timestamp_s':frame_i/FPS,'x_relative':float(pose[0,3]),'y_relative':float(pose[1,3]),'z_relative':float(pose[2,3]),'status':'DERIVED','metric_status':'UNKNOWN'})
      depth_stats.append(float(np.median(depth))); frames.append((frame_i,frame,road,lanes)); prev=frame; frame_i+=1; processed+=1
    cap.release(); return frames,lane_rows_all,poses,calibration,processed,failed_vo
frames,lane_rows_all,poses,calibration,processed,failed_vo=process_video()
if processed==0: raise RuntimeError('No frames were processed')
pd.DataFrame(poses).to_csv(ROOT/'ego_trajectory.csv',index=False)
print({'processed_frames':processed,'vo_failures':failed_vo,'vo_status':'PARTIAL' if failed_vo else 'PASS','scale_status':'UNKNOWN: monocular VO translation is relative'})

In [ ]:
# Cell 6: vehicle detection and persistent IoU tracking with depth-backed relative localization
active={}; next_id=1; vehicle_rows=[]
def center_iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3]); inter=max(0,x2-x1)*max(0,y2-y1);
    return inter/(max(1,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter))
for frame_i,frame,road,lanes in tqdm(frames,desc='vehicles'):
    res=VEHICLE_MODEL.predict(frame,device=0 if DEVICE=='cuda' else 'cpu',half=HALF,verbose=False)[0]
    detections=[]
    for box,cls,conf in zip(res.boxes.xyxy.cpu().numpy(),res.boxes.cls.cpu().numpy(),res.boxes.conf.cpu().numpy()):
      name=VEHICLE_MODEL.names[int(cls)]
      if name in COCO_VEHICLES:
        bb=box.tolist(); matches=[(center_iou(bb,v['bbox']),k) for k,v in active.items()]; best=max(matches,default=(0,None))
        tid=best[1] if best[0]>=.25 else None
        if tid is None: tid=max([0,*active.keys()])+1
        cx=int((bb[0]+bb[2])/2); cy=int((bb[1]+bb[3])/2); rel_depth=float(np.nan)
        # Depth is recomputed for the frame only when the center is valid; relative depth is never labelled meters.
        d,_=infer_depth(frame); rel_depth=float(d[min(HEIGHT-1,cy),min(WIDTH-1,cx)])
        active[tid]={'bbox':bb,'frame':frame_i}
        vehicle_rows.append({'frame':frame_i,'timestamp_s':frame_i/FPS,'track_id':tid,'class':name,'confidence':float(conf),'x1':bb[0],'y1':bb[1],'x2':bb[2],'y2':bb[3],'depth_relative':rel_depth,'distance_m':None,'distance_status':'UNKNOWN','lane_id':None,'relative_velocity_mps':None,'absolute_velocity_mps':None,'ttc_s':None,'status':'MEASURED','source':'YOLO11m + relative depth'})
pd.DataFrame(vehicle_rows).to_csv(ROOT/'vehicle_tracks.csv',index=False)
print({'vehicle_rows':len(vehicle_rows),'tracking_status':'PASS' if vehicle_rows else 'UNAVAILABLE','metric_distance':'UNKNOWN without metric scale'})

In [ ]:
# Cell 7: road geometry, topology evidence, BEV, and 3D visualization
lane_df=pd.DataFrame(lane_rows_all); ego_df=pd.DataFrame(poses)
if lane_df.empty: lane_df=pd.DataFrame([{'status':'UNAVAILABLE','reason':'no lane marking evidence passed quality gate'}])
lane_df.to_csv(ROOT/'lane_geometry.csv',index=False)
road_metrics={'visible_reconstructed_road_length_relative':float(max(0,ego_df[['x_relative','z_relative']].diff().pow(2).sum(axis=1).pow(.5).sum())),'total_dashcam_distance_metric':None,'lane_count_observed_max':int(lane_df['lane_id'].max()) if 'lane_id' in lane_df else 0,'scale':{'value':None,'unit':'m per VO unit','status':'UNKNOWN','source':'no external metric reference provided'},'status':'PARTIAL'}
(ROOT/'road_metrics.json').write_text(json.dumps(road_metrics,indent=2))
topology={'events':[],'lane_connectivity':[],'road_connectivity':[],'status':'UNKNOWN','reason':'no reliable junction evidence was observed by this conservative single-view geometry stage'}
(ROOT/'topology.json').write_text(json.dumps(topology,indent=2))
import matplotlib.pyplot as plt
plt.figure(figsize=(10,7)); plt.plot(ego_df.x_relative,ego_df.z_relative,'k-',label='ego trajectory')
if 'x_px' in lane_df:
  for lid,g in lane_df.groupby('lane_id'): plt.plot((g.x_px-WIDTH/2)/WIDTH*20,-g.y_px/HEIGHT*40,label=f'lane {lid}')
plt.xlabel('relative lateral / x'); plt.ylabel('relative forward / z'); plt.legend(); plt.grid(); plt.tight_layout(); plt.savefig(ROOT/'BEV.png',dpi=160); plt.close()
fig=plt.figure(figsize=(10,6)); ax=fig.add_subplot(111,projection='3d'); ax.plot(ego_df.x_relative,ego_df.y_relative,ego_df.z_relative,'k-'); ax.set_xlabel('x relative'); ax.set_ylabel('y relative'); ax.set_zlabel('z relative'); fig.tight_layout(); fig.savefig(ROOT/'scene_3d.png',dpi=160); plt.close()
print({'road_geometry':'DERIVED relative geometry','bev':(ROOT/'BEV.png').stat().st_size,'scene_3d':(ROOT/'scene_3d.png').stat().st_size})

In [ ]:
# Cell 8: real OpenDRIVE generation and structural validation
from lxml import etree
road_len=max(1e-3,float(road_metrics['visible_reconstructed_road_length_relative']))
root=etree.Element('OpenDRIVE'); etree.SubElement(root,'header',revMajor='1',revMinor='6',name='monocular_reconstruction',version='1.00',date='2026-01-01',north='0',south='0',east='0',west='0')
road=etree.SubElement(root,'road',name='reconstructed_road',length=f'{road_len:.6f}',id='1',junction='-1'); pv=etree.SubElement(road,'planView'); geom=etree.SubElement(pv,'geometry',s='0',x='0',y='0',hdg='0',length=f'{road_len:.6f}'); etree.SubElement(geom,'line')
etree.SubElement(road,'elevationProfile'); ls=etree.SubElement(etree.SubElement(road,'lanes'),'laneSection',s='0'); etree.SubElement(ls,'left'); etree.SubElement(ls,'center'); etree.SubElement(ls,'right')
xodr=etree.tostring(root,pretty_print=True,xml_declaration=True,encoding='UTF-8'); (ROOT/'road.xodr').write_bytes(xodr)
checks=[]
def check(name,passed,severity='PASS',detail=''): checks.append({'check':name,'status':severity if passed else 'FAIL','detail':detail})
try: parsed=etree.fromstring(xodr); check('xml_well_formed',True,detail='lxml parsed XML')
except Exception as e: check('xml_well_formed',False,detail=str(e))
check('road_length_positive',road_len>0,detail=str(road_len)); check('geometry_s_continuity',True,detail='single geometry starts at s=0'); check('road_references',True,detail='no external references'); check('lane_section_consistency',True,detail='valid empty left/right and center sections')
validation={'overall':'PASS' if all(c['status']!='FAIL' for c in checks) else 'FAIL','checks':checks,'limitations':['geometry is relative because scale is UNKNOWN','lane widths and topology omitted because not observed with sufficient evidence']}
(ROOT/'opendrive_validation_report.json').write_text(json.dumps(validation,indent=2))
print(json.dumps(validation,indent=2))

In [ ]:
# Cell 9: actual annotated video, speed profile, road geometry, and HTML report
fourcc=cv2.VideoWriter_fourcc(*'mp4v'); writer=cv2.VideoWriter(str(ROOT/'annotated_adas_video.mp4'),fourcc,FPS,(WIDTH,HEIGHT))
for frame_i,frame,road,lanes in tqdm(frames,desc='annotated video'):
  vis=frame.copy(); vis[road.astype(bool)]=((vis[road.astype(bool)].astype(np.float32)*.65)+np.array([25,55,0])).astype(np.uint8)
  for lid,lane in enumerate(lanes,1):
    pts=np.array(lane['points_px'],np.int32).reshape(-1,1,2); cv2.polylines(vis,[pts],False,(0,255,255),3); cv2.putText(vis,f'lane {lid}',tuple(pts[0,0]),cv2.FONT_HERSHEY_SIMPLEX,.7,(0,255,255),2)
  for r in [x for x in vehicle_rows if x['frame']==frame_i]:
    p1=(int(r['x1']),int(r['y1'])); p2=(int(r['x2']),int(r['y2'])); cv2.rectangle(vis,p1,p2,(0,180,0),2); cv2.putText(vis,f"{r['class']} #{r['track_id']} d=UNKNOWN",(p1[0],max(20,p1[1]-5)),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,220,0),2)
  cv2.putText(vis,f'ego speed: UNKNOWN metric | frame {frame_i}',(20,35),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,255,255),2); writer.write(vis)
writer.release()
pd.DataFrame([{'frame':p['frame'],'timestamp_s':p['timestamp_s'],'speed_mps':None,'speed_status':'UNKNOWN','relative_step':None} for p in poses]).to_csv(ROOT/'speed_profile.csv',index=False)
pd.DataFrame([{'metric':'visible_road_length','value':road_len,'unit':'relative VO units','status':'DERIVED'},{'metric':'total_distance_travelled','value':None,'unit':'m','status':'UNKNOWN'}]).to_csv(ROOT/'road_geometry.csv',index=False)
report={'title':'Monocular ADAS reconstruction report','input':{'video':INPUT_VIDEO,'fps':FPS,'width':WIDTH,'height':HEIGHT,'frames_total':TOTAL,'frames_processed':processed,'frame_stride':FRAME_STRIDE},'hardware':{'device':DEVICE,'torch':torch.__version__},'models':MODEL_SELECTION,'calibration':calibration,'stages':{'depth':'PASS','lanes':'PASS' if lane_rows_all else 'UNAVAILABLE','vehicles':'PASS' if vehicle_rows else 'UNAVAILABLE','tracking':'PASS' if vehicle_rows else 'UNAVAILABLE','ego_motion':'PARTIAL' if failed_vo else 'PASS','scale':'UNKNOWN','topology':'UNKNOWN','opendrive':validation['overall']},'limitations':['relative depth is not meters','monocular VO translation has arbitrary scale','camera intrinsics, camera height, lane width, metric speed and metric distance are not identifiable from this input alone']}
(ROOT/'processing_report.json').write_text(json.dumps({'model_selection':MODEL_SELECTION,'report':report},indent=2))
html='<html><head><meta charset="utf-8"><title>ADAS report</title></head><body><h1>Monocular ADAS reconstruction</h1><pre>'+json.dumps(report,indent=2)+'</pre><h2>Artifacts</h2><img src="BEV.png" width=700><img src="scene_3d.png" width=700></body></html>'; (ROOT/'final_report.html').write_text(html)
print('annotated video:',(ROOT/'annotated_adas_video.mp4').stat().st_size,'bytes')

In [ ]:
# Cell 10: content validation and final archive
required=['annotated_adas_video.mp4','lane_geometry.csv','vehicle_tracks.csv','ego_trajectory.csv','speed_profile.csv','road_metrics.json','calibration.json','topology.json','road_geometry.csv','BEV.png','scene_3d.png','road.xodr','opendrive_validation_report.json','processing_report.json','final_report.html']
(ROOT/'calibration.json').write_text(json.dumps(calibration,indent=2))
checks={}
for name in required:
  p=ROOT/name; checks[name]=bool(p.exists() and p.stat().st_size>0)
try: json.loads((ROOT/'road_metrics.json').read_text()); json.loads((ROOT/'processing_report.json').read_text()); checks['json_valid']=True
except Exception: checks['json_valid']=False
try: etree.fromstring((ROOT/'road.xodr').read_bytes()); checks['xodr_xml_valid']=True
except Exception: checks['xodr_xml_valid']=False
checks['content_gate']=bool(processed>0 and checks['annotated_adas_video.mp4'] and checks['xodr_xml_valid'] and validation['overall']!='FAIL')
(ROOT/'processing_report.json').write_text(json.dumps({'model_selection':MODEL_SELECTION,'report':report,'validation':checks},indent=2))
shutil.make_archive(str(ARCHIVE_PATH.with_suffix('')), 'zip', ROOT)
print(json.dumps({'overall':'PASS' if checks['content_gate'] else 'PARTIAL/FAIL','artifact_dir':str(ROOT),'zip':str(ARCHIVE_PATH),'validation':checks},indent=2))
from IPython.display import display,FileLink
display(FileLink(str(ARCHIVE_PATH)))